- Ce script charge d'abord un fichier CSV nommé Data_EHCVM_2021.csv, qui contient des données de ménages incluant les coordonnées GPS, et l'utilise pour chercher des correspondances avec des images satellites localisées dans un dossier spécifique. Les images satellites sont nommées avec des informations de latitude et de longitude, et le script analyse les noms de ces fichiers pour en extraire ces coordonnées. Ensuite, il compare ces valeurs extraites avec les colonnes gps__latitude et gps__longitude dans le fichier CSV pour identifier les ménages correspondant aux images disponibles.

- Pour chaque image ayant une correspondance, le script récupère les informations complètes du ménage depuis le fichier CSV et crée une entrée avec le nom de l'image en plus des autres données. Les colonnes sont ensuite réorganisées pour placer en première position le nom de l'image, suivi de colonnes clés (gps__latitude, gps__Longitude, grappe, region, hhweight, hhsize, pcexp) avant de compléter avec toutes les autres colonnes restantes.

- Enfin, le script sauvegarde le nouveau DataFrame dans un dossier spécifié. Si ce dossier n'existe pas, il est créé automatiquement. Le fichier résultant, nommé Data_EHCVM_2021_images.csv, contient donc une vue consolidée et organisée des informations de ménages associés à leurs images satellites.

In [1]:
import pandas as pd
import os
import glob

# Charger le fichier Data_EHCVM_2021.csv
data_path = r"D:\wealth_predict_2021\data\original_csv_file\Data_EHCVM_2021.csv"
data = pd.read_csv(data_path)

# Définir le chemin du dossier des images
images_folder = r"D:\wealth_predict_2021\data\downloaded\Image_satellite_EHCVM_2021_Zoom_18_Image_2024"

# Chercher les images avec plusieurs extensions possibles
image_extensions = ["*.jpeg", "*.png", "*.tif", "*.tiff"] 
images = []
for ext in image_extensions:
    images.extend(glob.glob(os.path.join(images_folder, ext)))

# Vérifier si des images ont été trouvées
if not images:
    print("Aucune image trouvée dans le dossier spécifié.")
else:
    # Créer une liste pour stocker les informations
    image_data = []

    # Extraire latitude et longitude des noms de fichier et associer avec les données de DataCIV3
    for image_path in images:
        # Extraire le nom de l'image
        image_name = os.path.basename(image_path)
        
        # Extraire latitude et longitude depuis le nom de l'image
        try:
            parts = image_name.split("_")
            latitude = float(parts[1])
            longitude = float(parts[3])
            
            # Filtrer Data_EHCVM_2021 pour correspondre les GPS
            matching_row = data[(data['gps__latitude'] == latitude) & (data['gps__longitude'] == longitude)]
            
            # Si une correspondance est trouvée, ajouter les informations au DataFrame
            if not matching_row.empty:
                # Ajouter le nom de l'image et les autres colonnes au DataFrame
                row_data = matching_row.iloc[0].to_dict()
                row_data["nom de l'image"] = image_name
                image_data.append(row_data)
                
        except (IndexError, ValueError):
            print(f"Erreur lors du traitement de l'image {image_name}. Nom de fichier non valide.")

    # Créer un DataFrame avec les informations des images
    image_df = pd.DataFrame(image_data)

    # Réorganiser les colonnes
    columns_order = ['nom de l\'image', 'gps__latitude', 'gps__longitude', 'grappe', 'region', 
                     'hhweight', 'hhsize', 'pcexp']
    remaining_columns = [col for col in image_df.columns if col not in columns_order]
    ordered_columns = columns_order + remaining_columns
    image_df = image_df[ordered_columns]

    # Définir le chemin du dossier de sauvegarde et créer le dossier s'il n'existe pas
    output_folder = r'D:\wealth_predict_2021\data\processed_csv'
    os.makedirs(output_folder, exist_ok=True)

    # Sauvegarder le DataFrame
    output_path = os.path.join(output_folder, "Data_EHCVM_2021_images.csv") 
    image_df.to_csv(output_path, index=False)

    print(f"DataFrame sauvegardé avec succès dans {output_path}")


DataFrame sauvegardé avec succès dans D:\wealth_predict_2021\data\processed_csv\Data_EHCVM_2021_images.csv


In [ ]:
image_df

In [ ]:
pd.read_csv(r"D:\wealth_predict_2021\data\processed_csv\Data_EHCVM_2021_images.csv")